# Experiment 1 — pooled analysis (modes A, B, C)

Set **`BASE_DIR`** to one folder that contains **`experiment_A.json`**, **`experiment_B.json`**, and **`experiment_C.json`**. The default layout uses repo-root **`experiment1_pooled/`** (you can symlink or copy the three files there). Figures are saved under **`outputs/`** next to this notebook (`notebooks/outputs/`).

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# --- User config: directory that holds experiment_A.json, experiment_B.json, experiment_C.json ---
_here = Path.cwd().resolve()
_cands = [
    _here / "experiment1_pooled",
    _here.parent / "experiment1_pooled",
]
BASE_DIR = next((p for p in _cands if p.is_dir()), _here.parent / "experiment1_pooled")

if (_here / "notebooks" / "experiment1_analysis.ipynb").is_file():
    NOTEBOOK_DIR = _here / "notebooks"
elif (_here / "experiment1_analysis.ipynb").is_file():
    NOTEBOOK_DIR = _here
else:
    NOTEBOOK_DIR = _here / "notebooks" if (_here / "notebooks").is_dir() else _here

OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Optional explicit override (uncomment and edit):
# BASE_DIR = Path("/path/to/dir_with_three_jsons").resolve()

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


def load_experiment(name: str) -> dict:
    p = BASE_DIR / name
    if not p.is_file():
        raise FileNotFoundError(f"Missing: {p}")
    with p.open(encoding="utf-8") as f:
        return json.load(f)


data_a = load_experiment("experiment_A.json")
data_b = load_experiment("experiment_B.json")
data_c = load_experiment("experiment_C.json")

In [ ]:
def onset_layer(ld_deltas):
    """First layer index where ld_delta <= -5; None if never."""
    arr = np.asarray(ld_deltas, dtype=float)
    for i, v in enumerate(arr):
        if v <= -5.0:
            return int(i)
    return None


def min_delta(pair):
    return float(np.min(pair["ld_delta_vs_clean_baseline_by_layer"]))


def category_label(category: str) -> str:
    return category.split("_")[0] if "_" in category else category.split()[0]


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_facecolor("white")
    ax.grid(True, alpha=0.35, linestyle="-")


MODE_COLORS = {"A": "purple", "B": "green", "C": "coral"}

In [ ]:
def summarize_mode(data: dict) -> dict:
    pairs = data["pairs"]
    n = len(pairs)
    worst = np.array([p["worst_layer_min_delta"] for p in pairs], dtype=float)
    onsets = []
    for p in pairs:
        o = onset_layer(p["ld_delta_vs_clean_baseline_by_layer"])
        if o is not None:
            onsets.append(o)
    onsets = np.array(onsets, dtype=float)
    baseline = np.array([p["baseline_ld_clean"] for p in pairs], dtype=float)
    mins = np.array([min_delta(p) for p in pairs], dtype=float)
    r, p_pearson = stats.pearsonr(baseline, mins)
    return {
        "n": n,
        "worst_min": int(np.nanmin(worst)),
        "worst_max": int(np.nanmax(worst)),
        "worst_mean": float(np.mean(worst)),
        "all_worst_ge_15": bool(np.all(worst >= 15)),
        "all_worst_ge_13": bool(np.all(worst >= 13)),
        "mean_onset": float(np.mean(onsets)) if len(onsets) else float("nan"),
        "n_onset_valid": len(onsets),
        "pearson_r": float(r),
        "pearson_p": float(p_pearson),
    }


for label, d in [("A", data_a), ("B", data_b), ("C", data_c)]:
    s = summarize_mode(d)
    print("=" * 60)
    print(f"Mode {label}  |  n_pairs = {s['n']}")
    print(
        f"  worst_layer_min_delta: min={s['worst_min']}, max={s['worst_max']}, mean={s['worst_mean']:.3f}"
    )
    print(f"  all worst_layer >= 15 ? {s['all_worst_ge_15']}")
    print(f"  all worst_layer >= 13 ? {s['all_worst_ge_13']}")
    print(
        f"  mean onset layer (first ld_delta<=-5): {s['mean_onset']:.3f}  (n_valid={s['n_onset_valid']}/{s['n']})"
    )
    print(
        f"  Pearson r(baseline_ld_clean, min_delta): r={s['pearson_r']:.4f}, p={s['pearson_p']:.4g}"
    )
print("=" * 60)

In [ ]:
# Figure 1 — mean ld_delta curves
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
x_layers = np.arange(data_a["n_layers"])

for ax, (mode, data) in zip(axes, [("A", data_a), ("B", data_b), ("C", data_c)]):
    pairs = data["pairs"]
    n = len(pairs)
    color = MODE_COLORS[mode]
    curves = []
    onsets = []
    for p in pairs:
        y = np.asarray(p["ld_delta_vs_clean_baseline_by_layer"], dtype=float)
        curves.append(y)
        ax.plot(x_layers, y, color=color, alpha=0.22, linewidth=0.9)
        o = onset_layer(p["ld_delta_vs_clean_baseline_by_layer"])
        if o is not None:
            onsets.append(o)
    mean_c = np.mean(np.stack(curves, axis=0), axis=0)
    ax.plot(x_layers, mean_c, color=color, linewidth=2.4, label="mean")
    ax.axhline(-5.0, color="0.35", linestyle="--", linewidth=1)
    ax.axvline(14.5, color="0.45", linestyle=":", linewidth=1)
    if onsets:
        mo = float(np.mean(onsets))
        ax.axvline(mo, color="0.25", linestyle="--", linewidth=1.2)
    style_axes(ax)
    ax.set_xlabel("layer")
    ax.set_ylabel(r"$\Delta$LD vs clean")
    ax.set_title(f"Mode {mode}  (n={n})")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig1_mean_ld_delta.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2 — worst layer histogram (layers 13–17)
layers_bins = np.arange(13, 18)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

for ax, (mode, data) in zip(axes, [("A", data_a), ("B", data_b), ("C", data_c)]):
    worst = np.array([p["worst_layer_min_delta"] for p in data["pairs"]], dtype=int)
    counts = [int(np.sum(worst == L)) for L in layers_bins]
    bars = ax.bar(layers_bins, counts, width=0.85, color=MODE_COLORS[mode], edgecolor="0.3", alpha=0.85)
    for L, c in zip(layers_bins, counts):
        if c > 0:
            ax.text(L, c + 0.08 * max(counts + [1]), str(c), ha="center", va="bottom", fontsize=9)
    mw = float(np.mean(worst))
    ax.axvline(mw, color="0.25", linestyle="--", linewidth=1.2)
    style_axes(ax)
    ax.set_xticks(layers_bins)
    ax.set_xlabel("worst_layer_min_delta")
    ax.set_ylabel("count")
    ax.set_title(f"Mode {mode}")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig2_worst_layer_hist.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 3 — baseline_ld_clean vs min(ld_delta)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

for ax, (mode, data) in zip(axes, [("A", data_a), ("B", data_b), ("C", data_c)]):
    pairs = data["pairs"]
    x = np.array([p["baseline_ld_clean"] for p in pairs], dtype=float)
    y = np.array([min_delta(p) for p in pairs], dtype=float)
    ax.scatter(x, y, c=MODE_COLORS[mode], s=36, alpha=0.85, edgecolors="0.3", linewidths=0.5)
    for p in pairs:
        ax.annotate(
            category_label(p["category"]),
            (p["baseline_ld_clean"], min_delta(p)),
            textcoords="offset points",
            xytext=(4, 3),
            fontsize=7,
            alpha=0.9,
        )
    lr = stats.linregress(x, y)
    xs = np.linspace(np.min(x), np.max(x), 100)
    ax.plot(xs, lr.intercept + lr.slope * xs, color="0.2", linewidth=1.5)
    r2 = lr.rvalue**2
    ax.text(
        0.05,
        0.95,
        f"r={lr.rvalue:.3f}\nr²={r2:.3f}\np={lr.pvalue:.3g}",
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.75", alpha=0.95),
    )
    style_axes(ax)
    ax.set_xlabel("baseline_ld_clean")
    ax.set_ylabel("min(ΔLD)")
    ax.set_title(
        f"Mode {mode}  |  r={lr.rvalue:.3f}, r²={r2:.3f}, p={lr.pvalue:.3g}"
    )

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig3_baseline_vs_min_delta.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 4 — total_swing vs min(ld_delta)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

for ax, (mode, data) in zip(axes, [("A", data_a), ("B", data_b), ("C", data_c)]):
    pairs = data["pairs"]
    x = np.array([p["total_swing"] for p in pairs], dtype=float)
    y = np.array([min_delta(p) for p in pairs], dtype=float)
    ax.scatter(x, y, c=MODE_COLORS[mode], s=36, alpha=0.85, edgecolors="0.3", linewidths=0.5)
    for p in pairs:
        ax.annotate(
            category_label(p["category"]),
            (p["total_swing"], min_delta(p)),
            textcoords="offset points",
            xytext=(4, 3),
            fontsize=7,
            alpha=0.9,
        )
    lr = stats.linregress(x, y)
    xs = np.linspace(np.min(x), np.max(x), 100)
    ax.plot(xs, lr.intercept + lr.slope * xs, color="0.2", linewidth=1.5)
    r2 = lr.rvalue**2
    ax.text(
        0.05,
        0.95,
        f"r={lr.rvalue:.3f}\nr²={r2:.3f}\np={lr.pvalue:.3g}",
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="0.75", alpha=0.95),
    )
    style_axes(ax)
    ax.set_xlabel("total_swing")
    ax.set_ylabel("min(ΔLD)")
    ax.set_title(
        f"Mode {mode}  |  r={lr.rvalue:.3f}, r²={r2:.3f}, p={lr.pvalue:.3g}"
    )

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fig4_total_swing_vs_min_delta.png", dpi=150, bbox_inches="tight")
plt.show()